CS74 Final Project: This project utilizes NBA statistics and a focus on binary classification, multi-class classification, and clustering to categorize NBA players. Raw player data (points, assists, steals, etc.) is used as input. In terms of labels, for binary classification, 4 levels of quality of player are used, for multiclass classification, 5 different tiers of salary are used, and for clustering, 5 different player positions are used.


In [1]:
import pandas as pd
import numpy as np
import unicodedata

## Load three csvs
salaries = pd.read_csv('NBA Player Stats and Salaries_2010-2025.csv')
advanced = pd.read_csv('Advanced.csv')
career   = pd.read_csv('Player Career Info.csv')

print(f"Salaries:  {salaries.shape}")
print(f"Advanced:  {advanced.shape}")
print(f"Career:    {career.shape}")

Salaries:  (7298, 31)
Advanced:  (33339, 30)
Career:    (5416, 11)


Need to merge these CSVs, each contain some of the data we need, we need to merge them into one CSV containing everything we need. First, names need to be normalized across datasets.

In [2]:
def normalize_name(s):
    """Strip accents, lowercase, remove punctuation for reliable joining."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return s.lower().replace('.', '').replace("'", '').strip()

salaries['_key'] = salaries['Player'].apply(normalize_name)
advanced['_key'] = advanced['player'].apply(normalize_name)
career['_key']   = career['player'].apply(normalize_name)

Here, we match rows by name, year, and team (we're using 2010 - 2025 data). Traded players are special cases, they appear three times in most datasets (stats with just Team 1, just Team 2, and over the whole year), so we must strip away those entries into one row of full-year data. We merge first by (name, year, team), then merge another time for traded players, using player ID instead.

In [33]:
salaries_keyed = salaries.copy()

# Single-team rows in advanced, match by (name, year, team)
# Handles all non-traded players AND correctly separates same-name collisions
adv_single = advanced_clean[
    advanced_clean['team'].str.len() == 3
][['_key', 'season', 'team', 'bpm', 'player_id']].rename(
    columns={'season': 'Year', 'team': 'Team'}
)

# Multi-team summary rows for traded players
adv_summary = advanced_clean[
    advanced_clean['team'].str.match(r'^\d+TM$', na=False)
][['_key', 'season', 'bpm', 'player_id']].rename(
    columns={'season': 'Year', 'bpm': 'bpm_summary', 'player_id': 'player_id_summary'}
)

# First merge: by exact (name, year, team)
df = salaries_keyed.merge(adv_single, on=['_key', 'Year', 'Team'], how='left')

# Second merge: fill in BPM/player_id for traded players via summary row
df = df.merge(adv_summary, on=['_key', 'Year'], how='left')
df['bpm'] = df['bpm'].fillna(df['bpm_summary'])
df['player_id'] = df['player_id'].fillna(df['player_id_summary'])
df = df.drop(columns=['bpm_summary', 'player_id_summary'])

# Career merge, now reliable because we use player_id, not name
career_by_id = career[['player_id', 'ht_in_in', 'wt', 'from']].rename(
    columns={'ht_in_in': 'height_in', 'wt': 'weight_lb', 'from': 'rookie_year'}
)
df = df.merge(career_by_id, on='player_id', how='left')

df['total_minutes'] = df['G'] * df['MP']
df['experience'] = df['Year'] - df['rookie_year']
df = df.drop(columns=['_key'])

print(f"Final df total rows:    {len(df)}")
print(f"Missing BPM:            {df['bpm'].isna().sum()}")
print(f"Missing height:         {df['height_in'].isna().sum()}")
print(f"Missing player_id:      {df['player_id'].isna().sum()}")
print(f"Duplicate player-years: {df.duplicated(subset=['Player', 'Year', 'Team']).sum()}")

# Test: same-name players preserved correctly with different player_ids
print("\nVerifying same-name collisions are correctly preserved:")
print(df[df['Player'].isin(['Chris Johnson', 'Tony Mitchell']) & df['Year'].isin([2013, 2014])][
    ['Player', 'Year', 'Team', 'player_id', 'bpm', 'Salary']
].sort_values(['Player', 'Year']))

Final df total rows:    7298
Missing BPM:            1
Missing height:         0
Missing player_id:      0
Duplicate player-years: 0

Verifying same-name collisions are correctly preserved:
             Player  Year Team  player_id   bpm  Salary
1745  Chris Johnson  2013  MEM  johnsch04   0.5   55718
1746  Chris Johnson  2013  MIN  johnsch03   1.6   55718
2135  Chris Johnson  2014  BOS  johnsch04  -1.2  320189
2113  Tony Mitchell  2014  DET  mitchto02   0.4  500000
2114  Tony Mitchell  2014  MIL  mitchto03  10.6  500000


Now we need to create labels. First, we need to filter by 250 minutes a season. End of the bench players have too much variance in their season long statistics, and are less important in this experiment. For binary classification, our labels are Rotation Player, Starter, Star, and Superstar. Maybe a bit broad for in-depth basketball analysis, but more than satisfactory for this project. To get these labels, we use Box Plus-Minus (BPM) Cutoffs. BPM is an all-encompassing stat that aims to quantify overall impact of an NBA player. For multi-class classification, we use Salary Tiers. Instead of actual dollar values, we're using within-season salary percentile; makes inflation easy to account for. Our final label, for clustering, is each players position. This is already in our data--pretty easy to extract.

In [34]:
# === FILTER & CREATE LABELS ===

# Apply 500+ total minutes filter
before = len(df)
df = df[df['total_minutes'] >= 250].copy()
print(f"After 250-minute filter: {len(df)} rows (dropped {before - len(df)})")


# --- Task 1 labels: four BPM binary cutoffs ---
df['bpm_rotation']  = (df['bpm'] >= -2.0).astype(int)
df['bpm_starter']   = (df['bpm'] >=  2.0).astype(int)
df['bpm_star']      = (df['bpm'] >=  4.0).astype(int)
df['bpm_superstar'] = (df['bpm'] >=  6.0).astype(int)

# --- Task 2 label: salary tier by within-season percentile ---
def assign_salary_tier(group):
    pct = group.rank(pct=True)
    return pd.cut(
        pct,
        bins=[0, 0.40, 0.70, 0.90, 0.98, 1.01],
        labels=[1, 2, 3, 4, 5],
        include_lowest=True,
    ).astype(int)

df['salary_tier'] = df.groupby('Year')['Salary'].transform(assign_salary_tier)

# --- Task 3 label: position is already there as 'Pos' ---
# Handle multi-position labels like "PG-SG" — keep the primary position
df['Pos_primary'] = df['Pos'].str.split('-').str[0]

# === Verify class balance ===
print("\n--- BPM binary cutoffs (positive class share) ---")
for col, label in [('bpm_rotation', 'Rotation'), ('bpm_starter', 'Starter'),
                   ('bpm_star', 'Star'),         ('bpm_superstar', 'Superstar')]:
    print(f"  {label:10s} (BPM >= ...): {df[col].mean():.1%}")

print("\n--- Salary tiers ---")
print(df['salary_tier'].value_counts().sort_index())

print("\n--- Position distribution ---")
print(df['Pos_primary'].value_counts())

After 500-minute filter: 6020 rows (dropped 1278)

--- BPM binary cutoffs (positive class share) ---
  Rotation   (BPM >= ...): 69.7%
  Starter    (BPM >= ...): 15.2%
  Star       (BPM >= ...): 5.4%
  Superstar  (BPM >= ...): 2.1%

--- Salary tiers ---
salary_tier
1    2399
2    1806
3    1203
4     482
5     130
Name: count, dtype: int64

--- Position distribution ---
Pos_primary
SG    1358
PF    1203
C     1169
PG    1164
SF    1126
Name: count, dtype: int64


In [39]:
keep_cols = [
    # Identifiers
    'Player', 'player_id', 'Year', 'Team',
    # Demographics
    'Age', 'height_in', 'weight_lb', 'experience',
    # Position
    'Pos_primary',
    # Volume
    'G', 'GS', 'MP', 'total_minutes',
    # Per-game stats
    'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%',
    'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV',
    'PF', 'PTS',
    # Labels
    'bpm', 'bpm_rotation', 'bpm_starter', 'bpm_star', 'bpm_superstar',
    'Salary', 'salary_tier',
]

df_clean = df[keep_cols].copy()
df_clean.to_csv('data/nba_clean.csv', index=False)

print(f"Saved {len(df_clean)} rows × {len(df_clean.columns)} columns to ../data/nba_clean.csv")
df_clean.head()

Saved 6020 rows × 42 columns to ../data/nba_clean.csv


,Player,player_id,Year,Team,Age,height_in,weight_lb,experience,Pos_primary,G,...,TOV,PF,PTS,bpm,bpm_rotation,bpm_starter,bpm_star,bpm_superstar,Salary,salary_tier
0,Kobe Bryant,bryanko01,2010,LAL,31,78.0,212.0,13,SG,73,...,3.2,2.6,27.0,4.1,1,1,1,0,23034375,5
1,Jermaine O'Neal,onealje01,2010,MIA,31,83.0,226.0,13,C,70,...,1.8,3.0,13.6,-0.1,1,0,0,0,23016000,5
2,Tracy McGrady,mcgratr01,2010,NYK,30,80.0,210.0,12,SG,30,...,1.5,1.3,8.2,-2.2,0,0,0,0,22843124,5
3,Tim Duncan,duncati01,2010,SAS,33,83.0,250.0,12,C,78,...,1.8,1.9,17.9,6.1,1,1,1,1,22183220,5
4,Shaquille O'Neal,onealsh01,2010,CLE,37,85.0,325.0,17,C,53,...,2.0,3.2,12.0,-0.2,1,0,0,0,21000000,5
